In [77]:
fact_orders = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

dim_users = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/dim_users"
)

In [78]:
from pyspark.sql.functions import *

customer_metrics = (
    fact_orders
    .groupBy("user_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("sales_amount").alias("total_spend"),
        avg("sales_amount").alias("avg_order_value")
    )
)

In [79]:
customer_segments = (
    customer_metrics
    .join(dim_users, "user_id")
)

In [80]:
from pyspark.sql.functions import when

customer_segments = customer_segments.withColumn(
    "customer_segment",
    when(
        (col("total_spend") > 50000) &
        (col("total_orders") > 20),
        "Premium Customer"
    )
    .when(
        col("total_orders") > 15,
        "Frequent Buyer"
    )
    .when(
        col("total_spend") < 5000,
        "Budget Customer"
    )
    .otherwise("Occasional Buyer")
)

In [81]:
customer_segments.groupBy(
    "customer_segment"
).count().show()

In [82]:
customer_segments.write.mode("overwrite").parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/customer_segments"
)